In [1]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000021820941E80>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021820942900>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="This year movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movies rating out of 10")
    story: str = Field(description="base line of the store in 2-3 lines")

In [4]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000021820941E80>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021820942900>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'type': 'number'}, 'story': {'description': 'base line 

In [6]:
response = model.invoke("Provide details about the movie Bhahubali the conclusion")

In [9]:
response = model_with_structure.invoke("Provide details about the movie Bhahubali the conclusion")

In [11]:
print(response)

title='Baahubali: The Conclusion' year=2017 director='S.S. Rajamouli' rating=8.2 story="The sequel follows Amarendra Baahubali's quest for justice after discovering a conspiracy against his kingdom. It delves into themes of truth, power, and redemption, with a complex narrative involving political intrigue and epic battles."


In [13]:
model_with_structure = model.with_structured_output(Movie,include_raw=True)
response = model_with_structure.invoke("Provide details about the movie Bhahubali the conclusion", include_raw=True)


In [14]:
print(response)

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "Bhahubali: The Conclusion." Let me see what I need to do here. The available tool is the Movie function, which requires specific parameters: title, year, director, rating, and story. \n\nFirst, I need to confirm the correct title. "Bhahubali: The Conclusion" is part of the Bahubali series. Wait, the official title is "Bahubali: The Conclusion," right? The user wrote "Bhahubali" with a \'B,\' but the correct title starts with \'Bh.\' I should stick with the correct title to avoid confusion.\n\nNext, the year. The movie was released in 2017. The director is S.S. Rajamouli. The rating—I think it\'s quite high. On IMDb, it\'s around 8.1. The story needs a brief summary. The Bahubali series is about the Mahishmati kingdom, and the conclusion wraps up the story of Bhallaladeva and Amarendra Baahubali. It\'s an epic fantasy with themes of revenge and redemption.\n\nI ne

### Nested Structure

In [18]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float| None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Bhahubali the conclusion")

print(response)

title='Bahubali 2: The Conclusion' year=2017 cast=[Actor(name='Prabhas', role='Baahubali'), Actor(name='Rana Daggubati', role='Amarendra Baahubali'), Actor(name='Tamannaah Bhatia', role='Devasena'), Actor(name='Sushmita Sen', role='Shivagami'), Actor(name='Anushka Shetty', role='Nala')] genres=['Action', 'Adventure', 'Drama'] budget=100.0


In [19]:
from typing_extensions import TypedDict, Annotated

class MovieDetails(TypedDict):
    """ A movie with details."""

    title: Annotated[str, ...,"The title of the movie"]
    year: Annotated[int,..., "The year the moviewas released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[str, ..., "The movie's rating out of 10"]

model_with_typedict= model.with_structured_output(MovieDetails)

response = model_with_typedict.invoke("Pleaseprovide the details of the movie avengers the end game")

response

{'director': 'Anthony Russo, Joe Russo',
 'rating': '8.4',
 'title': 'Avengers: Endgame',
 'year': 2019}

In [20]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float| None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about the movie Avengers")

response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genres': ['Action', 'Science Fiction', 'Adventure'],
 'title': 'Avengers',
 'year': 2012}

In [21]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

In [22]:
from pydantic import BaseModel,Field

from langchain.agents import create_agent

class contactinfo(BaseModel):
    """ contact information for a person """
    name: str = Field(description="the name of the person")
    email: str = Field(description="The email adress of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model = model,
    response_format=contactinfo
)

result = agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info fromjhon doe, john@example.com,(1234) 12345-678910"}]
})

In [23]:
result

{'messages': [HumanMessage(content='Extract contact info fromjhon doe, john@example.com,(1234) 12345-678910', additional_kwargs={}, response_metadata={}, id='c5527c29-25a7-4089-98b1-a211226f8315'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact information from the given text: "jhon doe, john@example.com,(1234) 12345-678910". \n\nFirst, I need to identify the different parts. The name is "jhon doe", but I notice that "jhon" might be a typo for "John". Should I correct it? The function parameters require the name as a string, so maybe it\'s better to input it as given unless instructed otherwise. Wait, the user might have made a typo, but since the task is to extract, not correct, I should probably use it as is.\n\nNext, the email is clearly "john@example.com". That\'s straightforward. The phone number is "(1234) 12345-678910". The format here is a bit unusual. Typically, phone numbers might be in formats like (123

In [24]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    ''' contact information for a person.'''

    name: str #The name of the person
    email: str #The email addressof the person
    phone: str #The phone number of the person

agent = create_agent(
    model=model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info fromjhon doe, john@example.com,(1234) 12345-678910"}]
})

result

{'messages': [HumanMessage(content='Extract contact info fromjhon doe, john@example.com,(1234) 12345-678910', additional_kwargs={}, response_metadata={}, id='45182ba3-82cc-44f3-819a-00e0b77fc964'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact information from the given text. The text is "jhon doe, john@example.com,(1234) 12345-678910". First, I need to identify the name, email, and phone number.\n\nThe name part is "jhon doe". I notice that "jhon" might be a typo for "John", but since the user wrote it as "jhon", I should probably keep it as is unless there\'s a correction. The email is clearly "john@example.com". The phone number is "(1234) 12345-678910". Let me check the phone number format. The area code is in parentheses, followed by a space, then the rest of the number with a hyphen. The function parameters require name, email, and phone. So I need to map these parts into the ContactInfo function. \n\nWait,